# Практикалық семинар: AG News деректерінде BERT моделін Fine-Tuning жасау

**Мақсаты:** Алдын ала үйретілген (pre-trained) модельді арнайы тапсырмаға (жаңалықтарды 4 категорияға бөлу) бейімдеу (fine-tuning).

**Деректер жиынтығы (Dataset):** AG News. Бұл жаңалықтар тақырыптарын қамтитын және 4 класқа бөлінген жиынтық:

1. World (Әлем)
2. Sports (Спорт)
3. Business (Бизнес)
4. Sci/Tech (Ғылым/Технология)


### 1-қадам: Қажетті кітапханаларды орнату

Google Colab немесе жергілікті Jupyter Notebook ортасында төмендегі кітапханаларды орнату қажет.

In [ ]:
# Қажетті кітапханаларды орнату
!pip install transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


### 2-қадам: Кітапханаларды импорттау

In [ ]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

# Ескерту: Егер GPU бар болса, оны қолдануды тексереміз
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Қолданылып жатқан құрылғы: {device}")

Қолданылып жатқан құрылғы: cuda


### 3-қадам: Деректерді жүктеу және шолу

AG News деректер жиынтығын Hugging Face Hub-тан жүктейміз.

In [ ]:
# Деректерді жүктеу
dataset = load_dataset("ag_news")

# Деректер құрылымын қарау
print(dataset)

# Мысал ретінде бір деректі шығару
print("\nМысал дерек:")
print(dataset['train'][0])

# Класс атауларын қарау (Label names)
labels = dataset['train'].features['label'].names
print(f"\nКластар: {labels}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

Мысал дерек:
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}

Кластар: ['World', 'Sports', 'Business', 'Sci/Tech']


### 4-қадам: Мәтінді токенизациялау (Tokenization)

Модель мәтінді тікелей түсінбейді, сандарға айналдыру керек. Ол үшін біз `DistilBERT` токенизаторын қолданамыз.

In [ ]:
# Токенизаторды жүктеу
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Токенизация функциясы
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Барлық деректерге map функциясын қолдану
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Жылдам эксперимент үшін деректер көлемін азайтуға болады (Семинар уақытын үнемдеу үшін)
# Егер толық оқытқыңыз келсе, бұл қатарларды алып тастаңыз
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(500))

print("Деректер токенизацияланды және дайын.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Деректер токенизацияланды және дайын.


### 5-қадам: Бағалау метрикасын дайындау

Модельдің сапасын тексеру үшін `accuracy` (дәлдік) метрикасын қолданамыз.

In [ ]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

### 6-қадам: Модельді жүктеу

Біз `DistilBERT` моделін жүктейміз және оған бізде 4 класс бар екенін айтамыз (`num_labels=4`).

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 7-қадам: Тренинг параметрлерін орнату және оқыту (Fine-tuning)

Бұл ең маңызды қадам. Біз `Trainer` класын қолданып, оқу процесін бастаймыз.

In [ ]:
training_args = TrainingArguments(
    output_dir="test_trainer",          # Нәтижелер сақталатын папка
    eval_strategy="epoch",        # Әр эпоха сайын тексеру
    save_strategy="epoch",              # Әр эпоха сайын сақтау
    learning_rate=2e-5,                 # Оқу жылдамдығы
    per_device_train_batch_size=16,     # Бір қадамдағы деректер саны (Batch size)
    per_device_eval_batch_size=16,
    num_train_epochs=3,                 # Эпоха саны (Толық деректерді неше рет көреді)
    weight_decay=0.01,
    load_best_model_at_end=True,        # Ең жақсы нәтиже көрсеткен модельді жүктеу
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,  # Егер толық деректер болса: tokenized_datasets["train"]
    eval_dataset=small_eval_dataset,    # Егер толық деректер болса: tokenized_datasets["test"]
    compute_metrics=compute_metrics,
)

# Оқытуды бастау
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.589038,0.838000
2,No log,0.428121,0.878000
3,No log,0.427971,0.864000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=189, training_loss=0.5682267688569569, metrics={'train_runtime': 192.0089, 'train_samples_per_second': 15.624, 'train_steps_per_second': 0.984, 'total_flos': 397416370176000.0, 'train_loss': 0.5682267688569569, 'epoch': 3.0})

### 8-қадам: Нәтижені бағалау

Оқыту аяқталған соң, модельдің тест жиынтығындағы дәлдігін тексереміз.

In [ ]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.4279705286026001, 'eval_accuracy': 0.864, 'eval_runtime': 8.5692, 'eval_samples_per_second': 58.348, 'eval_steps_per_second': 3.734, 'epoch': 3.0}


### 9-қадам: Болжам жасау (Inference)

Енді дайын модельді қолданып, кез келген жаңа мәтіннің қай категорияға жататынын анықтаймыз.

In [ ]:
# Жаңа мәтіндер
texts = [
    "Apple unveiled the new iPhone 15 with a titanium finish today.", # Sci/Tech болуы керек
    "Manchester United won the match against Liverpool in the last minute.", # Sports болуы керек
    "Oil prices dropped significantly due to market fluctuations." # Business болуы керек
]

# Токенизация
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)

# Болжам жасау
model.to(device)
with torch.no_grad():
    logits = model(**inputs).logits

# Ең жоғарғы ықтималдықты алу
predictions = torch.argmax(logits, dim=-1)

# Нәтижені шығару
id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

print("\nБолжам нәтижелері:")
for text, pred in zip(texts, predictions):
    print(f"Мәтін: {text} \n -> Болжам: {id2label[pred.item()]}\n")


Болжам нәтижелері:
Мәтін: Apple unveiled the new iPhone 15 with a titanium finish today. 
 -> Болжам: Sci/Tech

Мәтін: Manchester United won the match against Liverpool in the last minute. 
 -> Болжам: Sports

Мәтін: Oil prices dropped significantly due to market fluctuations. 
 -> Болжам: Business




### Қорытындылау сұрақтары:

1. Epoch (дәуір) санын көбейтсе не өзгереді?
2. `num_labels` параметрін өзгертсе не болады?
3. Басқа модельді (мысалы, `bert-base-uncased`) қалай қолдануға болады?
